# Sistema de Recomendación de Anime

# Parte 1: EDA y preprocesamiento

---

## Contenido 

1. Carga de datos
2. Exploración (EDA)
3. Preprocesamiento
4. Modelo Base (Baseline)
5. Guardado de splits


## 0. Imports


In [1]:
import kaggle
import shutil
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")


You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


NameError: name 'exit' is not defined

In [1]:
import kagglehub
import shutil
import os

# 1. Descarga a la caché por defecto de kagglehub
cache_path = kagglehub.dataset_download("CooperUnion/anime-recommendations-database")
print("Descargado en caché:", cache_path)

# 2. Definimos la ruta de destino (carpeta en tu directorio actual)
target_path = "./dataset_anime"

# 3. Copiamos los archivos de la caché a nuestro directorio
if not os.path.exists(target_path):
    shutil.copytree(cache_path, target_path)
    print(f"Archivos copiados exitosamente a: {target_path}")
else:
    print(f"La carpeta {target_path} ya existe. Los archivos ya están ahí.")

Descargado en caché: C:\Users\niaib\.cache\kagglehub\datasets\CooperUnion\anime-recommendations-database\versions\1
Archivos copiados exitosamente a: ./dataset_anime


# 1. Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Data exploration

In [ ]:
anime = "dataset_anime/anime.csv"
rating = "dataset_anime/rating.csv"

df_anime = pd.read_csv(anime)
df_rating = pd.read_csv(rating)

In [11]:
df_anime.sample(5)

,anime_id,name,genre,type,episodes,rating,members
3157,873,.hack//Roots,"Adventure, Drama, Fantasy, Game, Sci-Fi",TV,26,7.06,50480
11950,4350,Megami Kyouju,"Hentai, Horror, Military, Sci-Fi, Supernatural",OVA,2,5.85,1269
8948,28483,Ijime wa Zettai Warui!,"Drama, School, Slice of Life",OVA,1,7.67,46
9448,28577,Makehen de! Roku-nen San-kumi no Hanshin Daish...,"Drama, Historical, Kids",OVA,1,8.00,37
6452,216,Pia Carrot e Youkoso!!: Sayaka no Koi Monogatari,"Comedy, Romance",Movie,1,6.11,1804


In [10]:
df_rating.sample(5)

,user_id,anime_id,rating
6195746,57917,4898,9
4563451,43535,28297,10
7534961,70547,29758,9
5765260,53982,6862,7
1919800,18654,19363,9


In [14]:
print(f"Dimensiones del dataset anime: {df_anime.shape}")
print(f"\nDimensiones del dataset rating: {df_rating.shape}")

Dimensiones del dataset anime: (12294, 7)

Dimensiones del dataset rating: (7813737, 3)


In [20]:
# VALORES NULOS

print(f"Valores nulos de anime: \n{df_anime.isna().sum()}")

print(f"\nValores nulos de rating: \n{df_rating.isna().sum()}")

Valores nulos de anime: 
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

Valores nulos de rating: 
user_id     0
anime_id    0
rating      0
dtype: int64


In [21]:
# VALORES DUPLICADOS

print(f"Valores duplicados de anime: \n{df_anime.duplicated().sum()}")
print(f"Valores duplicados de rating: \n{df_rating.duplicated().sum()}")

Valores duplicados de anime: 
0
Valores duplicados de rating: 
1


In [22]:
df_anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [24]:
df_rating.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7813737 entries, 0 to 7813736
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   user_id   int64
 1   anime_id  int64
 2   rating    int64
dtypes: int64(3)
memory usage: 178.8 MB


In [44]:
# TIPOS DE DATOS
df_anime = df_anime.convert_dtypes()
df_anime.dtypes

anime_id             Int64
name        string[python]
genre       string[python]
type        string[python]
episodes    string[python]
rating             Float64
members              Int64
dtype: object

In [33]:
df_rating = df_rating.convert_dtypes()
df_rating.dtypes

user_id     Int64
anime_id    Int64
rating      Int64
dtype: object

# 3. Data Cleaning

In [37]:
df_anime_cp = df_anime.copy()
df_rating_cp = df_rating.copy()

In [38]:
df_anime_cp["genre"] = df_anime_cp["genre"].str.split(",")

In [ ]:
df_anime_cp.loc[df_anime_cp["episodes"] == "Unknown", "episodes"] = np.nan


In [45]:
df_anime_cp["episodes"] = df_anime_cp["episodes"].astype("float")

In [46]:
df_anime_cp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  Int64  
 1   name      12294 non-null  string 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  string 
 4   episodes  11954 non-null  float64
 5   rating    12064 non-null  Float64
 6   members   12294 non-null  Int64  
dtypes: Float64(1), Int64(2), float64(1), object(1), string(2)
memory usage: 708.5+ KB


In [ ]:
# VALORES NAN
print(df_anime_cp.isna().sum())

df_anime_cp["episodes"] = df_anime_cp["episodes"].fillna()

anime_id      0
name          0
genre        62
type         25
episodes    340
rating      230
members       0
dtype: int64
